# DP/SFA state-space nodal (SSN) for a series RLC

This notebook validates the DP-domain SSN components on a series RLC one-port
(R = 10 Ohm, L = 0.01 H, C = 0.002 F):

- the dedicated `dp.ph1.Full_Serial_RLC`,
- the generic `dp.ph1.GenericTwoTerminalVTypeSSN`, parametrized directly by
  continuous-time state-space matrices `(A, B, C, D)`,
- the generic `dp.ph1.GenericTwoTerminalITypeSSN` for a current-driven capacitor.

Each SSN component is compared against the same circuit built from classical
R, L, C elements in the DP domain, where the dynamic phasor envelopes must
agree. The DP-SSN envelope is reconstructed to the time domain and overlaid on
the EMT and EMT-SSN waveforms. The accuracy study (time-step and signal-frequency
sweeps) lives in the companion notebook `DP_SSN_RLC_accuracy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dpsimpy
from villas.dataprocessing.readtools import read_timeseries_csv
from villas.dataprocessing.timeseries import TimeSeries

In [ ]:
R = 10.0
L = 0.01
C = 0.002

f0 = 50.0
vref = 1.0 * np.exp(1j * np.deg2rad(-90.0))

time_step = 1e-4
final_time = 0.1

log_dir = "logs"

In [ ]:
def run(sim_name, system, domain, comp, attr):
    dpsimpy.Logger.set_log_dir(f"{log_dir}/{sim_name}")
    logger = dpsimpy.Logger(sim_name)
    logger.log_attribute("y", attr, comp)

    sim = dpsimpy.Simulation(sim_name, dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_domain(domain)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.add_logger(logger)
    sim.run()

    results = read_timeseries_csv(
        f"{log_dir}/{sim_name}/{sim_name}.csv", print_status=False
    )
    return results["y"]


def dp_node(name):
    return dpsimpy.dp.SimNode(name)


def emt_node(name):
    return dpsimpy.emt.SimNode(name, dpsimpy.PhaseType.Single)

## Series RLC across DP, EMT, and SSN

The same one-port is driven by a 50 Hz voltage source. The DP source frequency
is the offset from the carrier (0 = steady on the nominal carrier), while the
EMT source frequency is the absolute 50 Hz. `build_rlc` assembles the classical
network; `build_ssn` replaces it with the single `Full_Serial_RLC` component.

In [ ]:
def build_rlc(ph1, node, gnd, src_freq):
    n1, n2, n3 = node("n1"), node("n2"), node("n3")
    vs = ph1.VoltageSource("vs")
    vs.set_parameters(vref, src_freq)
    res = ph1.Resistor("r")
    res.set_parameters(R)
    ind = ph1.Inductor("l")
    ind.set_parameters(L)
    cap = ph1.Capacitor("c")
    cap.set_parameters(C)
    vs.connect([gnd, n1])
    res.connect([n1, n2])
    ind.connect([n2, n3])
    cap.connect([n3, gnd])
    return dpsimpy.SystemTopology(f0, [n1, n2, n3], [vs, res, ind, cap]), ind


def build_ssn(ph1, node, gnd, src_freq):
    n1 = node("n1")
    vs = ph1.VoltageSource("vs")
    vs.set_parameters(vref, src_freq)
    rlc = ph1.Full_Serial_RLC("rlc")
    rlc.set_parameters(R, L, C)
    vs.connect([gnd, n1])
    rlc.connect([n1, gnd])
    return dpsimpy.SystemTopology(f0, [n1], [vs, rlc]), rlc

In [ ]:
dp_gnd = dpsimpy.dp.SimNode.gnd
emt_gnd = dpsimpy.emt.SimNode.gnd

sys, comp = build_rlc(dpsimpy.dp.ph1, dp_node, dp_gnd, 0.0)
dp_rlc = run("DP_RLC_classical", sys, dpsimpy.Domain.DP, comp, "i_intf")

sys, comp = build_ssn(dpsimpy.dp.ph1, dp_node, dp_gnd, 0.0)
dp_ssn = run("DP_RLC_SSN", sys, dpsimpy.Domain.DP, comp, "i_intf")

sys, comp = build_rlc(dpsimpy.emt.ph1, emt_node, emt_gnd, f0)
emt_rlc = run("EMT_RLC_classical", sys, dpsimpy.Domain.EMT, comp, "i_intf")

sys, comp = build_ssn(dpsimpy.emt.ph1, emt_node, emt_gnd, f0)
emt_ssn = run("EMT_RLC_SSN", sys, dpsimpy.Domain.EMT, comp, "i_intf")

In [ ]:
dp_rlc_emt = dp_rlc.frequency_shift(f0)
dp_ssn_emt = dp_ssn.frequency_shift(f0)

plt.figure(figsize=(9, 4))
plt.plot(emt_rlc.time, emt_rlc.values, color="0.6", linewidth=3, label="EMT classical")
plt.plot(emt_ssn.time, emt_ssn.values, linestyle="--", label="EMT-SSN")
plt.plot(
    dp_rlc_emt.time,
    dp_rlc_emt.values,
    linestyle=":",
    label="DP classical (reconstructed)",
)
plt.plot(
    dp_ssn_emt.time, dp_ssn_emt.values, linestyle="-.", label="DP-SSN (reconstructed)"
)
plt.xlabel("time [s]")
plt.ylabel("inductor current [A]")
plt.title("Series RLC inductor current: classical vs SSN, EMT vs DP")
plt.legend()
plt.show()

The DP-SSN envelope must reproduce the classical DP envelope exactly, since
both are solved in the same DP network. The reconstructed DP-SSN waveform must
match the EMT-SSN waveform within trapezoidal discretization error.

In [ ]:
max_env_err = np.max(np.abs(dp_ssn.values - dp_rlc.values))
print("DP-SSN vs DP classical, max |envelope error|:", max_env_err)
assert max_env_err < 1e-9

rmse = TimeSeries.rmse(dp_ssn_emt, emt_ssn)
print("DP-SSN reconstructed vs EMT-SSN, RMSE:", rmse)
assert rmse < 1e-3

## Generic V-type from (A, B, C, D)

The series RLC is now declared directly by its state space, with states
`x = [v_C; i_L]`, input = port voltage, output = port current `i_L`. The result
must equal the dedicated `Full_Serial_RLC` envelope to machine precision.

In [ ]:
A = [[0.0, 1.0 / C], [-1.0 / L, -R / L]]
B = [[0.0], [1.0 / L]]
Cm = [[0.0, 1.0]]
D = [[0.0]]

n1 = dp_node("n1")
vs = dpsimpy.dp.ph1.VoltageSource("vs")
vs.set_parameters(vref, 0.0)
gen = dpsimpy.dp.ph1.GenericTwoTerminalVTypeSSN("rlc_gen")
gen.set_parameters(A, B, Cm, D)
vs.connect([dp_gnd, n1])
gen.connect([n1, dp_gnd])
sys = dpsimpy.SystemTopology(f0, [n1], [vs, gen])
gen_v = run("DP_RLC_genV", sys, dpsimpy.Domain.DP, gen, "i_intf")

max_err = np.max(np.abs(gen_v.values - dp_ssn.values))
print("generic V-type vs Full_Serial_RLC, max |envelope error|:", max_err)
assert max_err < 1e-9

## Generic I-type from (A, B, C, D)

A capacitor is expressed as a current-driven I-type one-port (input = port
current, output = port voltage) with `dv_C/dt = i_C / C`, so `A = 0`, `B = 1/C`,
`C = 1`, `D = 0`. It is placed in parallel with a resistor and excited by a
current source, and compared against the same circuit using a classical
capacitor.

In [ ]:
n1 = dp_node("n1")
cs = dpsimpy.dp.ph1.CurrentSource("cs")
cs.set_parameters(vref)
res = dpsimpy.dp.ph1.Resistor("r")
res.set_parameters(R)
cap = dpsimpy.dp.ph1.Capacitor("c")
cap.set_parameters(C)
cs.connect([dp_gnd, n1])
res.connect([n1, dp_gnd])
cap.connect([n1, dp_gnd])
sys = dpsimpy.SystemTopology(f0, [n1], [cs, res, cap])
rc_v = run("DP_RC_classical", sys, dpsimpy.Domain.DP, cap, "v_intf")

In [ ]:
Ai = [[0.0]]
Bi = [[1.0 / C]]
Ci = [[1.0]]
Di = [[0.0]]

n1 = dp_node("n1")
cs = dpsimpy.dp.ph1.CurrentSource("cs")
cs.set_parameters(vref)
res = dpsimpy.dp.ph1.Resistor("r")
res.set_parameters(R)
geni = dpsimpy.dp.ph1.GenericTwoTerminalITypeSSN("c_gen")
geni.set_parameters(Ai, Bi, Ci, Di)
cs.connect([dp_gnd, n1])
res.connect([n1, dp_gnd])
geni.connect([n1, dp_gnd])
sys = dpsimpy.SystemTopology(f0, [n1], [cs, res, geni])
geni_v = run("DP_RC_genI", sys, dpsimpy.Domain.DP, geni, "v_intf")

max_err = np.max(np.abs(geni_v.values - rc_v.values))
print("generic I-type vs classical capacitor, max |envelope error|:", max_err)
assert max_err < 1e-9

In [ ]:
rc_emt = rc_v.frequency_shift(f0)
geni_emt = geni_v.frequency_shift(f0)

plt.figure(figsize=(9, 4))
plt.plot(rc_emt.time, rc_emt.values, color="0.6", linewidth=3, label="classical R-C")
plt.plot(geni_emt.time, geni_emt.values, linestyle="--", label="generic I-type SSN")
plt.xlabel("time [s]")
plt.ylabel("capacitor voltage [V]")
plt.title("Current-driven capacitor: classical vs I-type SSN")
plt.legend()
plt.show()